# 06 - Ablation Evaluation for ClawTeam v3.1

This notebook evaluates the central claim of ClawTeam: **harness orchestration adds value, and role LoRA adapters add further value on top of the harness**.

Main groups:
- E1: single-agent baseline
- E2: four specialists, Round 1 only, no LoRA
- E3: four specialists, Round 1 + Round 2 critique, no LoRA
- E4: E3 + Qwen3 LoRA for surgeon and medical oncologist

Important fix: each group explicitly resets LoRA environment flags and clears the in-process LoRA cache, so E2/E3 cannot be polluted by cloud `.env` settings.


## Step 0: 环境准备

In [ ]:
import os, sys, asyncio, json, time
from pathlib import Path
from collections import defaultdict


def find_backend_dir(start: Path) -> Path:
    candidates = [start, *start.parents]
    for base in candidates:
        for candidate in (base, base / 'backend'):
            if (candidate / 'graph').is_dir() and (candidate / 'config').is_dir():
                return candidate.resolve()
    raise FileNotFoundError('Cannot locate backend directory from notebook cwd')


NOTEBOOK_DIR = Path.cwd().resolve()
BACKEND_DIR = find_backend_dir(NOTEBOOK_DIR)
EVAL_DIR = BACKEND_DIR / 'eval'
if str(BACKEND_DIR) not in sys.path:
    sys.path.insert(0, str(BACKEND_DIR))

os.environ.setdefault('HF_ENDPOINT', 'https://hf-mirror.com')

DATA_DIR = EVAL_DIR / 'datasets' / 'data'
RESULTS_DIR = EVAL_DIR / 'results'
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

print(f'Backend dir: {BACKEND_DIR}')
print(f'Data dir: {DATA_DIR}')
print(f'Results dir: {RESULTS_DIR}')


## Step 1: 加载评测集（限制规模 100-200 题）

In [ ]:
import pandas as pd
import random
import re

# Public benchmark first. TumorBoard cases are optional qualitative examples, not the main accuracy set.
# Recommended final paper run: ABLATION_EVAL_LIMIT=150 or 200.
TEST_LIMIT_PER_FILE = int(os.getenv('ABLATION_TEST_LIMIT', '2000'))
EVAL_LIMIT = int(os.getenv('ABLATION_EVAL_LIMIT', '150'))
EVAL_SEED = int(os.getenv('ABLATION_EVAL_SEED', '42'))
EVAL_SOURCE_ORDER = [s.strip() for s in os.getenv('ABLATION_SOURCES', 'MedQA,CMExam,MedBullets').split(',') if s.strip()]
DROP_TRAINING_OVERLAP = os.getenv('ABLATION_DROP_TRAINING_OVERLAP', 'true').lower() in ('1', 'true', 'yes')
INCLUDE_TUMOR_BOARD = os.getenv('ABLATION_INCLUDE_TUMOR_BOARD', 'false').lower() in ('1', 'true', 'yes')
TUMOR_BOARD_LIMIT = int(os.getenv('ABLATION_TUMOR_BOARD_LIMIT', '30'))
ALLOW_TUMOR_BOARD_ONLY = os.getenv('ABLATION_ALLOW_TUMOR_BOARD_ONLY', 'false').lower() in ('1', 'true', 'yes')


def read_jsonl(path: Path):
    with open(path, 'r', encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if line:
                yield json.loads(line)


def _clean_options(values) -> list[str]:
    return [str(x).strip() for x in values if str(x).strip()]


def normalize_options(item: dict) -> list[str]:
    options = item.get('options') or item.get('Options') or item.get('choices')
    if isinstance(options, dict):
        if all(k in options for k in ['A', 'B', 'C', 'D']):
            return _clean_options(options[k] for k in ['A', 'B', 'C', 'D'])
        if all(k in options for k in ['opa', 'opb', 'opc', 'opd']):
            return _clean_options(options[k] for k in ['opa', 'opb', 'opc', 'opd'])
        if isinstance(options.get('value'), list):
            return _clean_options(options['value'])
        return _clean_options(v for _, v in sorted(options.items()))
    if isinstance(options, list):
        return _clean_options(options)

    op_values = []
    for key in ['opa', 'opb', 'opc', 'opd', 'ope', 'A', 'B', 'C', 'D', 'E']:
        if key in item and item[key]:
            op_values.append(str(item[key]).strip())
    return op_values


def normalize_answer_idx(item: dict):
    for key in ['answer_idx', 'answer_index', 'correct_idx', 'cop', 'label']:
        if key in item and item[key] not in (None, ''):
            return item[key]
    answer = item.get('answer') or item.get('Answer') or item.get('correct_answer')
    if isinstance(answer, str):
        s = answer.strip().upper()
        if s[:1] in list('ABCDEFGH'):
            return s[:1]
    return None


def normalize_benchmark_item(item: dict, source: str, path: Path, idx: int) -> dict | None:
    question = (
        item.get('question') or item.get('Question') or item.get('stem') or
        item.get('prompt') or item.get('input') or item.get('query') or ''
    )
    question = str(question).strip()
    options = normalize_options(item)
    answer_text = str(
        item.get('answer_text') or item.get('answer') or item.get('Answer') or
        item.get('correct_answer') or item.get('final_decision') or item.get('long_answer') or ''
    ).strip()
    answer_idx = normalize_answer_idx(item)

    if not question:
        return None
    question_text = question
    if options:
        opt_text = '\n'.join(f'{chr(65+i)}. {o}' for i, o in enumerate(options))
        question_text = f'{question}\n\n{opt_text}'

    return {
        'id': f'{source}-{path.stem}-{idx}',
        'source': source,
        'benchmark_file': str(path.relative_to(DATA_DIR)),
        'question': question_text,
        'question_stem': question,
        'options': options,
        'answer': answer_text,
        'answer_idx': answer_idx,
        'raw': item,
    }


def collect_public_benchmark_cases() -> tuple[list[dict], list[dict]]:
    # Main accuracy uses held-out test splits only. Train/validation are intentionally excluded.
    specs = [
        ('MedQA', DATA_DIR / 'oncology', ['medqa_test_oncology.jsonl']),
        ('CMExam', DATA_DIR / 'oncology', ['cmexam_test_oncology.jsonl']),
        ('MedBullets', DATA_DIR / 'oncology', ['medbullets_test_oncology.jsonl']),
    ]
    cases = []
    inventory = []
    seen = set()
    for source, directory, filenames in specs:
        files = [directory / name for name in filenames if (directory / name).exists()]
        for path in files:
            rows = list(read_jsonl(path))
            added = 0
            mcq = 0
            for idx, item in enumerate(rows, 1):
                case = normalize_benchmark_item(item, source, path, idx)
                if case is None:
                    continue
                key = (case['source'], case['question_stem'][:300], str(case.get('answer_idx')), case.get('answer', '')[:100])
                if key in seen:
                    continue
                seen.add(key)
                cases.append(case)
                added += 1
                mcq += int(bool(case.get('options')))
                if added >= TEST_LIMIT_PER_FILE:
                    break
            inventory.append({
                'source': source,
                'file': str(path.relative_to(DATA_DIR)),
                'split': 'test',
                'rows': len(rows),
                'usable_cases': added,
                'mcq_cases': mcq,
            })
    return cases, inventory


def leak_normalize(text: str) -> str:
    text = str(text or '').lower()
    text = re.sub(r'\s+', '', text)
    text = re.sub(r'[^0-9a-z\u4e00-\u9fff]+', '', text)
    return text


def extract_training_user_text(row: dict) -> str:
    if isinstance(row.get('messages'), list):
        parts = [str(m.get('content', '')) for m in row['messages'] if m.get('role') == 'user']
        return '\n'.join(parts)
    for key in ['question', 'query', 'input', 'prompt', 'text']:
        if row.get(key):
            return str(row[key])
    return json.dumps(row, ensure_ascii=False)


def find_training_overlaps(cases: list[dict]) -> tuple[set[str], list[dict]]:
    training_files = [
        DATA_DIR / 'training' / 'surgeon_train.jsonl',
        DATA_DIR / 'training' / 'oncologist_train.jsonl',
    ]
    training_texts = []
    for path in training_files:
        if not path.exists():
            continue
        for i, row in enumerate(read_jsonl(path), 1):
            text = leak_normalize(extract_training_user_text(row))
            if len(text) >= 30:
                training_texts.append((path.name, i, text))

    leaks = []
    leaked_ids = set()
    for case in cases:
        stem = leak_normalize(case.get('question_stem') or case.get('question', ''))
        full = leak_normalize(case.get('question', ''))
        if len(stem) < 30 and len(full) < 30:
            continue
        for train_file, train_line, train_text in training_texts:
            match_type = None
            if stem and stem in train_text:
                match_type = 'stem_in_training'
            elif full and full in train_text:
                match_type = 'full_question_in_training'
            elif len(stem) >= 80 and train_text in stem:
                match_type = 'training_in_stem'
            if match_type:
                leaked_ids.add(case['id'])
                leaks.append({
                    'case_id': case['id'],
                    'source': case.get('source'),
                    'benchmark_file': case.get('benchmark_file'),
                    'training_file': train_file,
                    'training_line': train_line,
                    'match_type': match_type,
                })
                break
    return leaked_ids, leaks


def stratified_select(cases: list[dict], limit: int, source_order: list[str], seed: int) -> list[dict]:
    rng = random.Random(seed)
    groups = {source: [] for source in source_order}
    extra = []
    for case in cases:
        if case['source'] in groups:
            groups[case['source']].append(case)
        else:
            extra.append(case)
    for group in groups.values():
        rng.shuffle(group)
    rng.shuffle(extra)

    active_sources = [s for s in source_order if groups.get(s)]
    selected = []
    if active_sources:
        base_quota = max(1, limit // len(active_sources))
        for source in active_sources:
            take = min(base_quota, len(groups[source]), limit - len(selected))
            selected.extend(groups[source][:take])
            groups[source] = groups[source][take:]

    # Fill remaining slots round-robin so one large benchmark does not dominate by accident.
    while len(selected) < limit:
        progressed = False
        for source in active_sources:
            if groups[source] and len(selected) < limit:
                selected.append(groups[source].pop(0))
                progressed = True
        if not progressed:
            break
    if len(selected) < limit:
        selected.extend(extra[:limit - len(selected)])
    return selected[:limit]


def collect_tumor_board_cases() -> list[dict]:
    path = DATA_DIR / 'tumor_board_cases.jsonl'
    if not path.exists():
        return []
    cases = []
    for i, item in enumerate(read_jsonl(path), 1):
        cases.append({
            'id': item.get('id', f'TumorBoard-{i}'),
            'source': 'TumorBoard',
            'benchmark_file': str(path.relative_to(DATA_DIR)),
            'question': item.get('case', item.get('question', '')),
            'question_stem': item.get('case', item.get('question', '')),
            'options': [],
            'answer': item.get('reference_plan', item.get('expected_answer', item.get('answer', ''))),
            'answer_idx': None,
            'raw': item,
        })
        if len(cases) >= TUMOR_BOARD_LIMIT:
            break
    return cases


def load_eval_set():
    public_cases, inventory = collect_public_benchmark_cases()
    mcq_public = [c for c in public_cases if c.get('options')]
    freeform_public = [c for c in public_cases if not c.get('options')]

    leaked_ids, leaks = find_training_overlaps(mcq_public)
    pd.DataFrame(leaks).to_csv(RESULTS_DIR / 'training_overlap_report.csv', index=False, encoding='utf-8-sig')
    if leaks:
        print(f'[leakage] Found {len(leaks)} benchmark cases overlapping with LoRA training data.')
        if DROP_TRAINING_OVERLAP:
            mcq_public = [c for c in mcq_public if c['id'] not in leaked_ids]
            print(f'[leakage] Dropped overlaps. Remaining public MCQ cases: {len(mcq_public)}')
        else:
            print('[leakage] Keeping overlaps because ABLATION_DROP_TRAINING_OVERLAP=false.')
    else:
        print('[leakage] No overlap found against local LoRA training jsonl files, or training files are absent.')

    selected = stratified_select(mcq_public, EVAL_LIMIT, EVAL_SOURCE_ORDER, EVAL_SEED)
    tumor_board = collect_tumor_board_cases()
    if INCLUDE_TUMOR_BOARD:
        selected = selected + tumor_board

    pd.DataFrame(inventory).to_csv(RESULTS_DIR / 'benchmark_inventory.csv', index=False, encoding='utf-8-sig')
    selected_counts = pd.DataFrame([{'source': c['source'], 'benchmark_file': c.get('benchmark_file', '')} for c in selected])
    if len(selected_counts):
        selected_counts.groupby(['source', 'benchmark_file']).size().reset_index(name='selected_cases').to_csv(
            RESULTS_DIR / 'selected_eval_set.csv', index=False, encoding='utf-8-sig'
        )

    print('=== Benchmark inventory: held-out test splits only ===')
    if inventory:
        for row in inventory:
            print(f"{row['source']:10s} {row['file']:35s} rows={row['rows']:<5d} usable={row['usable_cases']:<5d} mcq={row['mcq_cases']:<5d}")
    else:
        print('[warn] No held-out public benchmark files found under eval/datasets/data/oncology/.')
    print(f'public test cases: {len(public_cases)}; public test MCQ cases after leakage policy: {len(mcq_public)}; public free-form cases: {len(freeform_public)}')
    print(f'selected eval cases: {len(selected)}; sources: {dict(pd.Series([c["source"] for c in selected]).value_counts()) if selected else {}}')
    print(f'tumor board qualitative cases available: {len(tumor_board)}; included in this run: {INCLUDE_TUMOR_BOARD}')

    if not selected:
        if ALLOW_TUMOR_BOARD_ONLY:
            print('[warn] No public MCQ benchmark cases selected; falling back to TumorBoard smoke test only.')
            selected = tumor_board[:TUMOR_BOARD_LIMIT]
        else:
            raise RuntimeError(
                'No public held-out MCQ benchmark cases selected. Run download_medqa.py/download_cmexam.py, '
                'or set ABLATION_ALLOW_TUMOR_BOARD_ONLY=true for a smoke test only.'
            )
    if len([c for c in selected if c.get('options')]) < 50:
        print('[warn] Fewer than 50 scored MCQ cases selected. This is weak for the final report; use ABLATION_EVAL_LIMIT=100~200.')
    return selected


test_cases = load_eval_set()
mcq_cases = [c for c in test_cases if c.get('options')]
print(f'eval cases: {len(test_cases)}, MCQ cases: {len(mcq_cases)}')
if test_cases:
    print(f'\nFirst sample:\n{json.dumps(test_cases[0], ensure_ascii=False, indent=2)[:700]}')


## Step 2: 准备评估器 — 4 种实验配置

In [ ]:
import re
import gc
from graph.coordinator import Coordinator
from graph.complexity_assessor import CaseComplexity
from config import get_settings
from graph.llm import build_llm_config_from_settings, get_llm

coordinator = None

LORA_BASE = '/root/autodl-tmp/Qwen3-4B-Instruct-2507'
LORA_SURGEON_PATH = 'eval/models/surgeon_qwen3_lora'
LORA_ONCOLOGIST_PATH = 'eval/models/oncologist_qwen3_lora'


def clear_lora_cache():
    """Clear the LoRA loader cache so experiment groups are isolated."""
    try:
        import eval.inference.load_lora_role as loader
        cache = getattr(loader, '_lora_cache', None)
        if isinstance(cache, dict):
            cache.clear()
    except Exception as exc:
        print(f'[warn] clear_lora_cache failed: {exc}')
    gc.collect()
    try:
        import torch
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
    except Exception:
        pass


def configure_lora(enabled: bool):
    """Explicitly configure LoRA before each experiment group."""
    os.environ['USE_LORA_SURGEON'] = 'true' if enabled else 'false'
    os.environ['USE_LORA_MEDICAL_ONCOLOGIST'] = 'true' if enabled else 'false'
    os.environ['LORA_SURGEON_BASE'] = LORA_BASE
    os.environ['LORA_SURGEON_PATH'] = LORA_SURGEON_PATH
    os.environ['LORA_MEDICAL_ONCOLOGIST_BASE'] = LORA_BASE
    os.environ['LORA_MEDICAL_ONCOLOGIST_PATH'] = LORA_ONCOLOGIST_PATH
    clear_lora_cache()
    print(f'LoRA enabled: {enabled}')


def reset_coordinator(use_lora: bool):
    global coordinator
    configure_lora(use_lora)
    coordinator = Coordinator(BACKEND_DIR)
    return coordinator


def build_eval_prompt(case: dict) -> str:
    """Normalize E1-E4 input and ask for a parseable final answer."""
    q = case['question'].strip()
    if case.get('options'):
        return (
            f"{q}\n\n"
            "Reason briefly, then end with exactly one final answer line:\n"
            "Final Answer: <A/B/C/D>\n"
            "The final line must contain one option letter only."
        )
    return q


def expected_letter(case: dict) -> str | None:
    options = case.get('options') or []
    expected = str(case.get('answer', '')).strip().lower()
    if expected and options:
        for i, option in enumerate(options):
            if str(option).strip().lower() == expected:
                return chr(65 + i)

    idx = case.get('answer_idx')
    if isinstance(idx, int):
        if 0 <= idx < len(options or range(26)):
            return chr(65 + idx)
        if options and 1 <= idx <= len(options):
            return chr(65 + idx - 1)
    if isinstance(idx, str) and idx.strip():
        s = idx.strip().upper()
        if len(s) == 1 and 'A' <= s <= 'Z':
            return s
        if s.isdigit():
            n = int(s)
            if 0 <= n < len(options or range(26)):
                return chr(65 + n)
            if options and 1 <= n <= len(options):
                return chr(65 + n - 1)
    return None

def extract_final_letter(predicted: str, options: list | None = None) -> str | None:
    """Parse only explicit final answers; avoid broad substring matches."""
    if not predicted:
        return None
    text = predicted.strip()
    patterns = [
        r'final\s*answer\s*[:\uff1a]\s*([A-H])\b',
        r'\u6700\u7ec8\u7b54\u6848\s*[:\uff1a]?\s*([A-H])\b',
        r'\u7b54\u6848\s*[:\uff1a]?\s*([A-H])\b',
        r'\u9009\u62e9\s*([A-H])\b',
        r'\u9009\u9879\s*([A-H])\b',
    ]
    for pattern in patterns:
        matches = re.findall(pattern, text, flags=re.IGNORECASE)
        if matches:
            return matches[-1].upper()

    tail = text[-500:]
    matches = re.findall(r'(?<![A-Za-z])([A-H])\s*[\.\u3001\)]', tail)
    if matches:
        return matches[-1].upper()
    return None


def append_final_answer_line(text: str, letter: str) -> str:
    """Append a parseable final answer line without changing the original rationale."""
    text = (text or '').strip()
    final_line = f'Final Answer: {letter}'
    if re.search(r'final\s*answer\s*[:\uff1a]\s*[A-H]\b', text, flags=re.IGNORECASE):
        return text
    return f'{text}\n\n{final_line}' if text else final_line


def format_options(case: dict) -> str:
    return '\n'.join(f'{chr(65 + i)}. {option}' for i, option in enumerate(case.get('options') or []))


async def normalize_mcq_output(raw_output: str, case: dict) -> tuple[str, dict]:
    """Use the same post-hoc MCQ answer-format contract for E1-E4.

    This is a scoring normalizer, not a second medical judge: it first parses the
    original output. Only when the output is not parseable does it ask the global
    LLM to map the existing conclusion to A/B/C/D without adding new reasoning.
    """
    info = {
        'normalization_used': False,
        'normalization_status': 'not_mcq',
        'normalizer_output': '',
    }
    if not case.get('options'):
        return raw_output, info

    parsed = extract_final_letter(raw_output, case.get('options'))
    if parsed:
        info['normalization_status'] = 'already_parseable'
        return append_final_answer_line(raw_output, parsed), info

    info['normalization_used'] = True
    settings = get_settings()
    llm = get_llm(build_llm_config_from_settings(settings, temperature=0.0, streaming=False))
    normalizer_prompt = (
        "You are an MCQ answer formatter, not a medical decision maker.\n"
        "Do not solve the question again. Do not add evidence or change the rationale.\n"
        "Map the provided system output to exactly one option letter only if the output supports it.\n"
        "Return exactly one line: Final Answer: <A/B/C/D>.\n"
        "If the output does not support any option, return exactly: Final Answer: UNKNOWN."
    )
    user_prompt = (
        f"Options:\n{format_options(case)}\n\n"
        f"System output to format:\n{raw_output}"
    )
    try:
        response = await llm.ainvoke([
            {'role': 'system', 'content': normalizer_prompt},
            {'role': 'user', 'content': user_prompt},
        ])
        normalizer_output = getattr(response, 'content', '').strip()
        info['normalizer_output'] = normalizer_output
        letter = extract_final_letter(normalizer_output, case.get('options'))
        if letter:
            info['normalization_status'] = 'normalized'
            return append_final_answer_line(raw_output, letter), info
        info['normalization_status'] = 'failed_to_parse_normalizer'
        return raw_output, info
    except Exception as exc:
        info['normalization_status'] = f'normalizer_error:{exc!r}'
        return raw_output, info


async def run_e1_single_agent(case):
    settings = get_settings()
    llm = get_llm(build_llm_config_from_settings(settings, temperature=0.3, streaming=False))
    response = await llm.ainvoke([
        {'role': 'system', 'content': 'You are a medical exam assistant. Answer oncology MCQs and end with Final Answer: <letter>.'},
        {'role': 'user', 'content': build_eval_prompt(case)},
    ])
    return getattr(response, 'content', '').strip(), None


async def run_e2_no_debate(case):
    session = await coordinator.consult(
        case=build_eval_prompt(case),
        force_complexity=CaseComplexity.MODERATE,
        skip_round2=True,
    )
    return session.final_decision, session


async def run_e3_with_debate(case):
    session = await coordinator.consult(
        case=build_eval_prompt(case),
        force_complexity=CaseComplexity.COMPLEX,
        skip_round2=False,
    )
    return session.final_decision, session


async def run_e4_full(case):
    session = await coordinator.consult(
        case=build_eval_prompt(case),
        force_complexity=CaseComplexity.COMPLEX,
        skip_round2=False,
    )
    return session.final_decision, session


EXPERIMENTS = {
    'E1': {'name': 'E1_single', 'config': 'single-agent baseline', 'use_lora': False, 'fn': run_e1_single_agent},
    'E2': {'name': 'E2_no_debate', 'config': '4 specialists, Round 1 only, no LoRA', 'use_lora': False, 'fn': run_e2_no_debate},
    'E3': {'name': 'E3_debate', 'config': '4 specialists + Round 2 critique, no LoRA', 'use_lora': False, 'fn': run_e3_with_debate},
    'E4': {'name': 'E4_lora_heterogeneous', 'config': 'E3 + surgeon/oncologist LoRA + role-specific API overrides if configured', 'use_lora': True, 'fn': run_e4_full},
}

def configured_backend_snapshot() -> dict:
    """Record intended heterogeneous backend configuration from env."""
    return {
        'surgeon': {
            'expected': 'local_lora_when_enabled',
            'base': os.getenv('LORA_SURGEON_BASE', LORA_BASE),
            'path': os.getenv('LORA_SURGEON_PATH', LORA_SURGEON_PATH),
        },
        'medical_oncologist': {
            'expected': 'local_lora_when_enabled',
            'base': os.getenv('LORA_MEDICAL_ONCOLOGIST_BASE', LORA_BASE),
            'path': os.getenv('LORA_MEDICAL_ONCOLOGIST_PATH', LORA_ONCOLOGIST_PATH),
        },
        'pathologist_text': {
            'expected': 'text_api_when_no_image',
            'provider': os.getenv('PATHOLOGIST_LLM_PROVIDER', os.getenv('LLM_PROVIDER', 'global')),
            'model': os.getenv('PATHOLOGIST_LLM_MODEL', os.getenv('LLM_MODEL', 'global')),
        },
        'pathologist_vision': {
            'expected': 'vision_api_only_when_image_attachment_exists',
            'enabled': os.getenv('PATHOLOGIST_VISION_ENABLED', 'true'),
            'provider': os.getenv('PATHOLOGIST_VISION_PROVIDER', 'bailian'),
            'model': os.getenv('PATHOLOGIST_VISION_MODEL', 'qwen3-vl-plus'),
        },
        'radiation_oncologist': {
            'expected': 'strong_text_api',
            'provider': os.getenv('RADIATION_ONCOLOGIST_LLM_PROVIDER', os.getenv('LLM_PROVIDER', 'global')),
            'model': os.getenv('RADIATION_ONCOLOGIST_LLM_MODEL', os.getenv('LLM_MODEL', 'global')),
        },
    }


print('Experiment setup loaded: isolated LoRA flags, strict answer parsing, detailed logs.')
print(json.dumps(configured_backend_snapshot(), ensure_ascii=False, indent=2))


## Step 3: Evaluation helpers - strict answer parsing and detailed logs


In [ ]:
def score_prediction(predicted: str, case: dict) -> dict:
    exp = expected_letter(case)
    pred = extract_final_letter(predicted, case.get('options'))
    scored = exp is not None
    return {
        'expected_letter': exp,
        'predicted_letter': pred,
        'scored': scored,
        'correct': bool(scored and pred == exp),
    }


def collect_model_calls(session) -> list[dict]:
    """Collect per-role backend traces: api vs lora, round, and status."""
    if session is None:
        return []
    rows = []
    for attr in ['round1_opinions', 'round2_opinions']:
        for op in getattr(session, attr, []) or []:
            calls = getattr(op, 'tool_calls', None) or []
            if not calls:
                rows.append({
                    'type': 'model_call',
                    'role': getattr(op.role, 'value', str(op.role)),
                    'round': getattr(op, 'round_num', None),
                    'backend': 'unknown',
                    'status': 'missing_trace',
                })
            for call in calls:
                row = dict(call)
                row.setdefault('role', getattr(op.role, 'value', str(op.role)))
                row.setdefault('round', getattr(op, 'round_num', None))
                rows.append(row)
    return rows


def model_call_rows(calls: list[dict]) -> list[dict]:
    return [
        c for c in calls
        if c.get('type') == 'model_call' or c.get('backend') in ('lora', 'unknown') or str(c.get('backend', '')).startswith(('api', 'vision'))
    ]


def summarize_model_calls(calls: list[dict]) -> dict:
    model_rows = model_call_rows(calls)
    roles = sorted({c.get('role', '') for c in model_rows if c.get('role')})
    round1_roles = sorted({c.get('role', '') for c in model_rows if c.get('round') == 1})
    round2_roles = sorted({c.get('role', '') for c in model_rows if c.get('round') == 2})
    backends = sorted({str(c.get('backend', 'unknown')) for c in model_rows if c.get('backend')})
    text_api_calls = sum(1 for c in model_rows if str(c.get('backend', '')).startswith('api'))
    vision_api_calls = sum(1 for c in model_rows if str(c.get('backend', '')).startswith('vision'))
    lora_calls = sum(1 for c in model_rows if c.get('backend') == 'lora')
    failed_calls = sum(1 for c in model_rows if c.get('status') != 'ok')
    no_image_events = sum(1 for c in calls if c.get('type') == 'attachment_status' and c.get('status') == 'no_image')
    vision_disabled_events = sum(1 for c in calls if c.get('type') == 'attachment_status' and c.get('status') == 'vision_disabled')
    return {
        'model_calls': len(model_rows),
        'api_calls': text_api_calls + vision_api_calls,
        'text_api_calls': text_api_calls,
        'vision_api_calls': vision_api_calls,
        'lora_calls': lora_calls,
        'failed_calls': failed_calls,
        'no_image_events': no_image_events,
        'vision_disabled_events': vision_disabled_events,
        'roles_seen': '|'.join(roles),
        'round1_roles': '|'.join(round1_roles),
        'round2_roles': '|'.join(round2_roles),
        'backend_families': '|'.join(backends),
    }


async def evaluate_method(exp_key: str, cases: list[dict]):
    """Run one experiment group and keep per-case details."""
    spec = EXPERIMENTS[exp_key]
    reset_coordinator(use_lora=spec['use_lora'])

    correct = 0
    scored_total = 0
    latencies = []
    sessions = []
    details = []
    group_calls = []

    for i, case in enumerate(cases):
        start = time.monotonic()
        result = ''
        raw_result = ''
        session = None
        error = ''
        normalization_info = {
            'normalization_used': False,
            'normalization_status': 'not_run',
            'normalizer_output': '',
        }
        try:
            result, session = await spec['fn'](case)
            raw_result = result
            if session is not None:
                sessions.append(session)
        except Exception as exc:
            error = repr(exc)
            print(f'  [{i+1}/{len(cases)}] {spec["name"]} failed: {exc}')
        elapsed = time.monotonic() - start
        latencies.append(elapsed)
        if not error:
            result, normalization_info = await normalize_mcq_output(result, case)

        calls = collect_model_calls(session)
        if exp_key == 'E1':
            calls = [{
                'type': 'model_call',
                'role': 'single_agent',
                'round': 0,
                'backend': 'api:baseline',
                'status': 'failed' if error else 'ok',
            }]
        group_calls.extend(calls)
        call_summary = summarize_model_calls(calls)

        score = score_prediction(result, case)
        if score['scored']:
            scored_total += 1
            correct += int(score['correct'])

        details.append({
            'method': exp_key,
            'method_name': spec['name'],
            'case_id': case.get('id', i + 1),
            'source': case.get('source', ''),
            'expected_letter': score['expected_letter'],
            'predicted_letter': score['predicted_letter'],
            'correct': score['correct'],
            'scored': score['scored'],
            'latency_s': elapsed,
            'error': error,
            **call_summary,
            **normalization_info,
            'model_call_trace': json.dumps(calls, ensure_ascii=False),
            'question': case.get('question', '')[:500],
            'answer': case.get('answer', ''),
            'raw_prediction': raw_result,
            'prediction': result,
        })

        if (i + 1) % 5 == 0 or i + 1 == len(cases):
            acc = correct / scored_total if scored_total else 0
            group_trace = summarize_model_calls(group_calls)
            print(
                f'  [{i+1}/{len(cases)}] {spec["name"]}: '
                f'scored={scored_total}, acc={acc:.3f}, '
                f'api={group_trace["api_calls"]}, lora={group_trace["lora_calls"]}, '
                f'vision={group_trace["vision_api_calls"]}, roles={group_trace["roles_seen"]}'
            )

    revision_rates = [s.revision_rate for s in sessions if getattr(s, 'round2_opinions', None)]
    disagreement_counts = [s.disagreement_count for s in sessions if getattr(s, 'round2_opinions', None)]
    group_trace = summarize_model_calls(group_calls)

    return {
        'method': spec['name'],
        'config': spec['config'],
        'total': len(cases),
        'scored_total': scored_total,
        'correct': correct,
        'accuracy': correct / scored_total if scored_total else 0,
        'avg_latency_s': sum(latencies) / len(latencies) if latencies else 0,
        'sessions': sessions,
        'details': details,
        'avg_revision_rate': sum(revision_rates) / len(revision_rates) if revision_rates else 0,
        'avg_disagreement_count': sum(disagreement_counts) / len(disagreement_counts) if disagreement_counts else 0,
        **group_trace,
    }


## Step 4: 跑主实验（4 组）

In [ ]:
# Main 4-group experiment. Each group resets LoRA flags and cache.
results = {}
all_details = []

for exp_key in ['E1', 'E2', 'E3', 'E4']:
    print(f'=== {exp_key}: {EXPERIMENTS[exp_key]["config"]} ===')
    results[exp_key] = await evaluate_method(exp_key, test_cases)
    all_details.extend(results[exp_key]['details'])
    print(
        f'{exp_key} accuracy: {results[exp_key]["accuracy"]:.3f} '
        f'({results[exp_key]["correct"]}/{results[exp_key]["scored_total"]}), '
        f'latency: {results[exp_key]["avg_latency_s"]:.2f}s\n'
    )


<!-- merged into Step 4 runner above -->


<!-- merged into Step 4 runner above -->


<!-- merged into Step 4 runner above -->


## Step 5: 主实验汇总表（论文 Table 1）

In [ ]:
import pandas as pd

summary = pd.DataFrame([
    {
        'experiment': k,
        'config': v['config'],
        'scored_cases': v['scored_total'],
        'correct': v['correct'],
        'accuracy': f"{v['accuracy']:.3f}",
        'avg_latency_s': f"{v['avg_latency_s']:.2f}",
        'model_calls': v.get('model_calls', 0),
        'api_calls': v.get('api_calls', 0),
        'text_api_calls': v.get('text_api_calls', 0),
        'vision_api_calls': v.get('vision_api_calls', 0),
        'lora_calls': v.get('lora_calls', 0),
        'failed_calls': v.get('failed_calls', 0),
        'roles_seen': v.get('roles_seen', ''),
        'backend_families': v.get('backend_families', ''),
        'avg_revision_rate': f"{v['avg_revision_rate']:.3f}",
        'avg_disagreement_count': f"{v['avg_disagreement_count']:.2f}",
        'gain_vs_E1': f"{(v['accuracy'] - results['E1']['accuracy']) * 100:+.1f}%",
    }
    for k, v in results.items()
])
print('\n=== Main ablation results ===')
print(summary.to_string(index=False))

summary.to_csv(RESULTS_DIR / 'main_ablation.csv', index=False, encoding='utf-8-sig')
pd.DataFrame(all_details).to_csv(RESULTS_DIR / 'main_ablation_details.csv', index=False, encoding='utf-8-sig')
print(f'\nSaved: {RESULTS_DIR / "main_ablation.csv"}')
print(f'Details: {RESULTS_DIR / "main_ablation_details.csv"}')
print('Check api_calls/lora_calls/roles_seen/backend_families to verify all specialists were actually invoked.')

backend_audit_rows = []
for row in all_details:
    try:
        trace = json.loads(row.get('model_call_trace') or '[]')
    except Exception:
        trace = []
    for call in model_call_rows(trace):
        backend_audit_rows.append({
            'method': row.get('method'),
            'case_id': row.get('case_id'),
            'role': call.get('role', ''),
            'round': call.get('round', ''),
            'backend': call.get('backend', 'unknown'),
            'status': call.get('status', ''),
            'image_count': call.get('image_count', 0),
        })

backend_audit_df = pd.DataFrame(backend_audit_rows)
if not backend_audit_df.empty:
    backend_audit_summary = (
        backend_audit_df
        .groupby(['method', 'role', 'backend', 'status'], dropna=False)
        .size()
        .reset_index(name='calls')
        .sort_values(['method', 'role', 'backend'])
    )
else:
    backend_audit_summary = pd.DataFrame(columns=['method', 'role', 'backend', 'status', 'calls'])

print('\n=== Model backend audit ===')
print(backend_audit_summary.to_string(index=False))
backend_audit_df.to_csv(RESULTS_DIR / 'model_backend_audit_details.csv', index=False, encoding='utf-8-sig')
backend_audit_summary.to_csv(RESULTS_DIR / 'model_backend_audit.csv', index=False, encoding='utf-8-sig')
print(f'Backend audit: {RESULTS_DIR / "model_backend_audit.csv"}')


## Step 6: 子消融 1 — 训练 vs 不训练（外科 / 内科）

In [ ]:
print('=== Sub-ablation 1: training vs no training ===')
print('Use E4 - E3 as the incremental value of surgeon/oncologist LoRA within the same harness.')
training_gain = results['E4']['accuracy'] - results['E3']['accuracy']
print(f'Training gain (E4 - E3): {training_gain * 100:+.1f}%')
print('Note: LoRA is used only by surgeon and medical_oncologist. Pathologist and radiation_oncologist stay API + prompt.')


## Step 7: 子消融 2 — 复杂度评估方法对比

In [ ]:
from graph.complexity_assessor import assess_complexity

print('=== Complexity assessor comparison: distribution only ===')
routing_rows = []
for method in ['llm', 'bert', 'keyword']:
    counts = defaultdict(int)
    ok = 0
    for case in test_cases[:30]:
        try:
            decision = await assess_complexity(case['question'], method=method)
            counts[decision.level.value] += 1
            ok += 1
        except Exception as exc:
            counts['error'] += 1
            print(f'  {method} failed: {exc}')
    routing_rows.append({'method': method, 'ok': ok, **dict(counts)})

routing_df = pd.DataFrame(routing_rows).fillna(0)
print(routing_df.to_string(index=False))
routing_df.to_csv(RESULTS_DIR / 'complexity_methods.csv', index=False, encoding='utf-8-sig')
print('No gold complexity labels are available, so this is not reported as routing accuracy.')


## Step 8: Harness-focused complex-case subset


In [ ]:
from graph.complexity_assessor import assess_by_keyword

case_by_id = {case.get('id'): case for case in test_cases}
complex_case_ids = {
    case.get('id')
    for case in test_cases
    if assess_by_keyword(case.get('question', '')).level == CaseComplexity.COMPLEX
}

all_details_df = pd.DataFrame(all_details)
complex_details = all_details_df[all_details_df['case_id'].isin(complex_case_ids)].copy()

if complex_details.empty:
    print('No rule-enhanced complex cases found in selected MCQ set. Consider increasing ABLATION_EVAL_LIMIT or including TumorBoard cases.')
    complex_subset_summary = pd.DataFrame()
else:
    complex_subset_summary = (
        complex_details
        .groupby(['method', 'method_name'], dropna=False)
        .agg(
            cases=('case_id', 'nunique'),
            scored=('scored', 'sum'),
            correct=('correct', 'sum'),
            avg_latency_s=('latency_s', 'mean'),
            avg_revision_rate=('avg_revision_rate', 'mean'),
            api_calls=('api_calls', 'sum'),
            lora_calls=('lora_calls', 'sum'),
            vision_api_calls=('vision_api_calls', 'sum'),
        )
        .reset_index()
    )
    complex_subset_summary['accuracy'] = complex_subset_summary.apply(
        lambda row: row['correct'] / row['scored'] if row['scored'] else 0,
        axis=1,
    )
    complex_subset_summary = complex_subset_summary[
        ['method', 'method_name', 'cases', 'scored', 'correct', 'accuracy', 'avg_latency_s', 'api_calls', 'lora_calls', 'vision_api_calls']
    ]
    print('\n=== Harness-focused complex subset ===')
    print(f'complex cases selected: {len(complex_case_ids)} / {len(test_cases)}')
    print(complex_subset_summary.to_string(index=False, formatters={'accuracy': '{:.3f}'.format, 'avg_latency_s': '{:.2f}'.format}))

complex_details.to_csv(RESULTS_DIR / 'complex_subset_details.csv', index=False, encoding='utf-8-sig')
complex_subset_summary.to_csv(RESULTS_DIR / 'complex_subset_summary.csv', index=False, encoding='utf-8-sig')
print(f'Complex subset summary: {RESULTS_DIR / "complex_subset_summary.csv"}')


## Step 9: Sub-ablation 3 - Round count accuracy vs cost


In [ ]:
round_comp = pd.DataFrame([
    {
        'rounds': '1 (independent)',
        'experiment': 'E2',
        'accuracy': f"{results['E2']['accuracy']:.3f}",
        'avg_latency_s': f"{results['E2']['avg_latency_s']:.2f}",
        'avg_revision_rate': '-',
        'relative_cost': '1x',
    },
    {
        'rounds': '2 (independent + critique)',
        'experiment': 'E3',
        'accuracy': f"{results['E3']['accuracy']:.3f}",
        'avg_latency_s': f"{results['E3']['avg_latency_s']:.2f}",
        'avg_revision_rate': f"{results['E3']['avg_revision_rate']:.3f}",
        'relative_cost': f'{results["E3"]["avg_latency_s"] / max(results["E2"]["avg_latency_s"], 1e-6):.1f}x',
    },
])
print('\n=== Round count comparison ===')
print(round_comp.to_string(index=False))
round_comp.to_csv(RESULTS_DIR / 'round_comparison.csv', index=False, encoding='utf-8-sig')


## Step 10: Paper figures (matplotlib)


In [ ]:
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['font.sans-serif'] = ['DejaVu Sans']
matplotlib.rcParams['axes.unicode_minus'] = False

fig, ax = plt.subplots(figsize=(10, 6))
names = ['E1\nSingle', 'E2\nRoles', 'E3\nDebate', 'E4\nLoRA']
accs = [results[k]['accuracy'] for k in ['E1', 'E2', 'E3', 'E4']]
colors = ['#8A8F98', '#3B82F6', '#F59E0B', '#10B981']
bars = ax.bar(names, accs, color=colors)
for bar, acc in zip(bars, accs):
    ax.text(bar.get_x() + bar.get_width()/2, acc + 0.01, f'{acc:.3f}',
            ha='center', fontsize=11, fontweight='bold')
ax.set_ylabel('Accuracy on scored MCQ cases', fontsize=12)
ax.set_title('ClawTeam Main Ablation: Harness and LoRA Contributions', fontsize=14)
ax.set_ylim(0, max(max(accs) * 1.2, 0.1))
ax.grid(axis='y', linestyle='--', alpha=0.5)
plt.tight_layout()
plt.savefig(RESULTS_DIR / 'main_ablation.png', dpi=150)
plt.show()
print(f'\nFigure saved: {RESULTS_DIR / "main_ablation.png"}')


## Step 11: Final report JSON


In [ ]:
def to_jsonable(obj):
    """Convert pandas/numpy scalars into JSON-safe Python values."""
    try:
        import numpy as np
        if isinstance(obj, np.generic):
            return obj.item()
    except Exception:
        pass
    if isinstance(obj, dict):
        return {str(k): to_jsonable(v) for k, v in obj.items()}
    if isinstance(obj, (list, tuple, set)):
        return [to_jsonable(v) for v in obj]
    if not isinstance(obj, (str, bytes)):
        try:
            if pd.isna(obj):
                return None
        except Exception:
            pass
    return obj


source_counts = dict(pd.Series([c.get('source', 'unknown') for c in test_cases]).value_counts()) if test_cases else {}
benchmark_files = sorted({c.get('benchmark_file', '') for c in test_cases if c.get('benchmark_file')})

report = {
    'timestamp': time.strftime('%Y-%m-%d %H:%M:%S'),
    'benchmark_policy': {
        'main_accuracy_uses': 'held-out public oncology MCQ benchmarks only',
        'included_sources': EVAL_SOURCE_ORDER,
        'eval_limit': EVAL_LIMIT,
        'seed': EVAL_SEED,
        'drop_training_overlap': DROP_TRAINING_OVERLAP,
        'tumor_board_included': INCLUDE_TUMOR_BOARD,
        'notes': [
            'Training scripts exclude test splits by default; existing checkpoints are guarded by the overlap report.',
            'training_overlap_report.csv records and, by default, drops any overlap with local LoRA training jsonl files.',
            'TumorBoard cases are qualitative unless explicitly included; they are not scored as MCQ accuracy.',
        ],
    },
    'heterogeneous_team': configured_backend_snapshot(),
    'eval_set_size': len(test_cases),
    'scored_mcq_size': len([c for c in test_cases if expected_letter(c)]),
    'source_counts': source_counts,
    'benchmark_files': benchmark_files,
    'main_ablation': {
        k: {
            'config': v['config'],
            'accuracy': v['accuracy'],
            'correct': v['correct'],
            'scored_total': v['scored_total'],
            'avg_latency_s': v['avg_latency_s'],
            'avg_revision_rate': v['avg_revision_rate'],
            'avg_disagreement_count': v['avg_disagreement_count'],
            'model_calls': v.get('model_calls', 0),
            'api_calls': v.get('api_calls', 0),
            'text_api_calls': v.get('text_api_calls', 0),
            'vision_api_calls': v.get('vision_api_calls', 0),
            'lora_calls': v.get('lora_calls', 0),
            'failed_calls': v.get('failed_calls', 0),
            'roles_seen': v.get('roles_seen', ''),
            'backend_families': v.get('backend_families', ''),
        }
        for k, v in results.items()
    },
    'model_backend_audit': backend_audit_summary.to_dict(orient='records') if 'backend_audit_summary' in globals() else [],
    'complex_subset': complex_subset_summary.to_dict(orient='records') if 'complex_subset_summary' in globals() and not complex_subset_summary.empty else [],
    'key_findings': {
        'role_decomposition_gain_E2_vs_E1': results['E2']['accuracy'] - results['E1']['accuracy'],
        'debate_gain_E3_vs_E2': results['E3']['accuracy'] - results['E2']['accuracy'],
        'lora_gain_E4_vs_E3': results['E4']['accuracy'] - results['E3']['accuracy'],
        'total_gain_E4_vs_E1': results['E4']['accuracy'] - results['E1']['accuracy'],
    },
    'caveats': [
        'MCQ accuracy uses the same answer-format normalizer for E1-E4; the normalizer maps existing outputs to A/B/C/D and does not add medical reasoning.',
        'Public MCQ benchmarks do not evaluate pathology image understanding.',
        'Pathologist vision API is used only when image attachments are provided; no-image cases are text-only pathology opinions.',
        'Radiation oncologist can use a role-specific strong text API when RADIATION_ONCOLOGIST_LLM_* env vars are configured.',
        'TumorBoard free-form cases should be evaluated by rubric or LLM judge separately.',
        'Complexity assessor comparison reports distribution only because no gold complexity labels are available.',
        'Model backend audit records api/lora/vision backend per role; E2-E4 should include four specialist roles.',
    ],
}

report = to_jsonable(report)
with open(RESULTS_DIR / 'final_report.json', 'w', encoding='utf-8') as f:
    json.dump(report, f, ensure_ascii=False, indent=2)

print(json.dumps(report, ensure_ascii=False, indent=2))
